# 23c_calibrate_threshold — Platt 확률보정 + 임계값 튜닝 (신규)

**한 줄 요약:** 학습된 앙상블은 그대로 두고, **검증셋으로 ① 확률을 보정(Platt) ② MCC 최대 임계값을 선택** 한 뒤 test에서 before/after를 비교.
**조정 대상:** ① 확률→확률 시그모이드 함수(파라미터 2개) ② 판정 컷(숫자 1개). **둘 다 재학습 아님(후처리).**
**출력:** `data/calibration_threshold.png` (보정곡선·임계값-MCC·혼동행렬 before/after).

### 보정+임계값 실행
검증셋에서 Platt·임계값을 정하고 test에 적용해 비교.

In [ ]:
import os
while not os.path.isdir('data') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
print('작업 폴더:', os.getcwd())

import numpy as np, pandas as pd, pickle
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import matthews_corrcoef, confusion_matrix, brier_score_loss, f1_score
from sklearn.calibration import calibration_curve
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt

B=pickle.load(open("data/HSD17B13_repr_ensemble.pkl","rb")); TOP,REPS,PIPES=B["top"],B["reps"],B["pipelines"]
feat=pd.read_csv("data/HSD17B13_final_training_1to1_v2.csv"); mem=pd.read_csv("data/HSD17B13_rebalanced_membership.csv")
df=mem.merge(feat.drop(columns=["potency"]),on="canonical_smiles",how="left")
def ep(sp):
    return np.mean([PIPES[f"{m}|{r}"].predict_proba(df.loc[df.split==sp,REPS[r]].to_numpy())[:,1] for m,r in TOP],0)
vy=df.loc[df.split=="val","potency"].to_numpy();  vp=ep("val")
ty=df.loc[df.split=="test","potency"].to_numpy(); tp=ep("test")
src=df.loc[df.split=="test","source"].to_numpy()

def summ(y,p,thr,tag):
    pred=(p>=thr).astype(int); tn,fp,fn,tpv=confusion_matrix(y,pred,labels=[0,1]).ravel()
    print(f"[{tag}] thr={thr:.2f} | MCC {matthews_corrcoef(y,pred):.3f} | F1 {f1_score(y,pred):.3f} | "
          f"Brier {brier_score_loss(y,p):.3f} | TP{tpv} TN{tn} FP{fp} FN{fn}")
    for g in ["active","decoy","real_inactive"]:
        mm=src==g; acc=(pred[mm]==1).mean() if g=="active" else (pred[mm]==0).mean()
        print(f"     {g:14s} 정답률 {acc:.2f}")
    return matthews_corrcoef(y,pred),(tn,fp,fn,tpv)

print("===== BEFORE (원확률, 임계값 0.5) =====")
mcc_b,cm_b=summ(ty,tp,0.5,"BEFORE")

# --- Platt 보정: 검증셋에서 확률→확률 시그모이드 맞춤 ---
platt=LogisticRegression(C=1e6,solver="lbfgs").fit(vp.reshape(-1,1),vy)
cal=lambda p: platt.predict_proba(p.reshape(-1,1))[:,1]
vpc, tpc = cal(vp), cal(tp)

# --- 임계값 튜닝: 보정된 '검증'에서 MCC 최대 컷 ---
ths=np.linspace(0.05,0.95,91); vmccs=[matthews_corrcoef(vy,(vpc>=t).astype(int)) for t in ths]
best_t=float(ths[int(np.argmax(vmccs))])
print(f"\n검증셋 최적 임계값(MCC 최대) = {best_t:.2f}")

print("\n===== AFTER (Platt 보정 + 튜닝 임계값) =====")
mcc_a,cm_a=summ(ty,tpc,best_t,"AFTER")
print(f"\n>>> test MCC: {mcc_b:.3f} -> {mcc_a:.3f} | Brier: {brier_score_loss(ty,tp):.3f} -> {brier_score_loss(ty,tpc):.3f} (낮을수록 보정 좋음)")

# --- 그림 ---
fig,ax=plt.subplots(2,2,figsize=(13,11)); fig.suptitle("Calibration (Platt) + Threshold tuning — before/after",fontsize=14)
# A) calibration before/after (test)
for p,lab,mk in [(tp,"before",("o-","tab:blue")),(tpc,"after (Platt)",("s-","tab:orange"))]:
    fr,mpv=calibration_curve(ty,p,n_bins=8,strategy="uniform"); ax[0,0].plot(mpv,fr,mk[0],color=mk[1],label=lab)
ax[0,0].plot([0,1],[0,1],"k--",alpha=.5,label="perfect"); ax[0,0].set_title("A) Calibration curve (test)")
ax[0,0].set_xlabel("mean predicted prob"); ax[0,0].set_ylabel("fraction positive"); ax[0,0].legend(); ax[0,0].grid(alpha=.3)
# B) MCC vs threshold on val
ax[0,1].plot(ths,vmccs); ax[0,1].axvline(best_t,color="r",ls="--",label=f"best={best_t:.2f}")
ax[0,1].axvline(0.5,color="gray",ls=":",label="default 0.5")
ax[0,1].set_title("B) MCC vs threshold (validation)"); ax[0,1].set_xlabel("threshold"); ax[0,1].set_ylabel("MCC"); ax[0,1].legend(); ax[0,1].grid(alpha=.3)
# C/D confusion before/after
for k,(cm,ttl) in enumerate([(cm_b,"C) Confusion BEFORE (thr 0.5)"),(cm_a,f"D) Confusion AFTER (thr {best_t:.2f})")]):
    a=ax[1,k]; tn,fp,fn,tpv=cm; M=np.array([[tn,fp],[fn,tpv]])
    a.imshow(M,cmap="Oranges")
    for (i,j),v in np.ndenumerate(M): a.text(j,i,str(v),ha="center",va="center",fontsize=15,color="black")
    a.set_xticks([0,1]);a.set_xticklabels(["pred inact","pred act"]);a.set_yticks([0,1]);a.set_yticklabels(["actual inact","actual act"])
    a.set_title(ttl)
plt.tight_layout(rect=[0,0,1,0.97]); plt.savefig("data/calibration_threshold.png",dpi=130,bbox_inches="tight")
print("\n저장: data/calibration_threshold.png")
